In [ ]:
"""
03_financial_model.ipynb — финансовая модель Surf Coffee
Проект: Анализ открытия кофейни Surf Coffee рядом с ВШЭ

Вставляйте блоки между метками # ЯЧЕЙКА N в отдельные ячейки Jupyter/Colab.

Что считаем:
- Выручка: трафик × конверсия × средний чек × дни работы
- Издержки: аренда, ФОТ, COGS, маркетинг, операционные расходы
- Точка безубыточности (BEP)
- P&L на 12 месяцев
- Сценарии: пессимистичный / базовый / оптимистичный
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import os

matplotlib.rcParams["font.family"] = "DejaVu Sans"
matplotlib.rcParams["axes.unicode_minus"] = False

os.makedirs("data/processed", exist_ok=True)
os.makedirs("visuals", exist_ok=True)

# Форматирование чисел
def fmt(value):
    return f"{value:,.0f} ₽".replace(",", " ")

print("Готовы считать финансовую модель.")

In [ ]:
# ─── Расположение ──────────────────────────────────────────
# Покровский бульвар, рядом с корпусом ВШЭ
# Помещение ~80 кв.м (стандарт для кофейни с посадкой 25–30 мест)

# ─── ВЫРУЧКА ────────────────────────────────────────────────
# Из опроса:
# - средний чек ~ 234 ₽ (95% CI: 224–245 ₽)
# - Surf Coffee должна работать чуть выше рынка → 260 ₽
AVG_CHECK = 270                  # средний чек с учётом еды (кофе + выпечка)
AVG_DRINK_PRICE = 240            # средняя цена напитка (без еды)
DRINKS_PER_DAY_BASE = 240        # базовый сценарий — кол-во чеков в день

# ─── РАБОЧИЕ ДНИ ────────────────────────────────────────────
DAYS_PER_MONTH = 30
WORKING_DAYS_PER_YEAR = 360       # работаем почти каждый день

# ─── СТРУКТУРА ЧЕКА ─────────────────────────────────────────
# Доля кофе в выручке (с едой ~ 65/35)
COFFEE_REVENUE_SHARE = 0.65

# ─── СЕБЕСТОИМОСТЬ (COGS) ───────────────────────────────────
# Кофе: себестоимость ~25–30% от продажной цены
# Еда: себестоимость ~35–40%
COFFEE_COGS_PCT = 0.28
FOOD_COGS_PCT = 0.38

# ─── ПОСТОЯННЫЕ РАСХОДЫ В МЕСЯЦ ─────────────────────────────
# Аренда — Покровский бульвар, ~70 кв.м, ~4500 ₽/м² → 315 000 ₽
RENT_PER_MONTH = 315_000

# ФОТ:
# - 2 бариста посменно × 70 000 ₽   = 140 000 ₽
# - 2 кассира/универсала × 55 000 ₽ = 110 000 ₽
# - 1 управляющий × 100 000 ₽       = 100 000 ₽
# - Налоги/взносы ~30% сверху
PAYROLL_GROSS = 350_000
PAYROLL_TAXES = PAYROLL_GROSS * 0.30
PAYROLL_TOTAL = PAYROLL_GROSS + PAYROLL_TAXES  # = 455 000 ₽

# Прочие постоянные:
MARKETING = 60_000          # SMM, акции, флаеры для студентов
UTILITIES = 35_000          # коммуналка, интернет, эквайринг (фикс часть)
LICENSES_OTHER = 25_000     # лицензии, обслуживание ККМ, мелочи
FRANCHISE_FEE_PCT = 0.05    # роялти Surf Coffee — 5% от выручки

# ─── СТАРТОВЫЕ ИНВЕСТИЦИИ ───────────────────────────────────
# (нужны не в P&L, но важны для окупаемости — payback)
STARTUP_INVESTMENT = 4_500_000  # ремонт, оборудование, мебель, паушальный взнос, депозит

print("Исходные параметры заданы.")
print(f"Средний чек:                 {fmt(AVG_CHECK)}")
print(f"Базовый трафик в день:       {DRINKS_PER_DAY_BASE} чеков")
print(f"Стартовые инвестиции:        {fmt(STARTUP_INVESTMENT)}")
print(f"Постоянные расходы в месяц:  {fmt(RENT_PER_MONTH + PAYROLL_TOTAL + MARKETING + UTILITIES + LICENSES_OTHER)}")

In [ ]:
def calculate_monthly_pnl(checks_per_day: int, avg_check: int = AVG_CHECK,
                          days: int = DAYS_PER_MONTH) -> dict:
    """Рассчитывает P&L за месяц для заданного трафика."""

    # ВЫРУЧКА
    revenue = checks_per_day * avg_check * days
    coffee_revenue = revenue * COFFEE_REVENUE_SHARE
    food_revenue = revenue * (1 - COFFEE_REVENUE_SHARE)

    # COGS
    coffee_cogs = coffee_revenue * COFFEE_COGS_PCT
    food_cogs = food_revenue * FOOD_COGS_PCT
    total_cogs = coffee_cogs + food_cogs

    # Валовая прибыль
    gross_profit = revenue - total_cogs

    # Операционные расходы
    franchise_fee = revenue * FRANCHISE_FEE_PCT
    fixed_costs = (RENT_PER_MONTH + PAYROLL_TOTAL + MARKETING +
                   UTILITIES + LICENSES_OTHER)
    total_opex = fixed_costs + franchise_fee

    # Операционная прибыль
    operating_profit = gross_profit - total_opex
    margin = operating_profit / revenue if revenue else 0

    return {
        "checks_per_day": checks_per_day,
        "revenue": revenue,
        "coffee_revenue": coffee_revenue,
        "food_revenue": food_revenue,
        "total_cogs": total_cogs,
        "gross_profit": gross_profit,
        "franchise_fee": franchise_fee,
        "fixed_costs": fixed_costs,
        "total_opex": total_opex,
        "operating_profit": operating_profit,
        "margin_pct": margin * 100,
    }


# Базовый сценарий
base = calculate_monthly_pnl(DRINKS_PER_DAY_BASE)

print("=" * 60)
print(f"P&L (базовый сценарий, {DRINKS_PER_DAY_BASE} чеков/день)")
print("=" * 60)
print(f"Выручка в месяц:                {fmt(base['revenue'])}")
print(f"  - Кофе и напитки:             {fmt(base['coffee_revenue'])}")
print(f"  - Еда:                        {fmt(base['food_revenue'])}")
print(f"Себестоимость (COGS):           {fmt(base['total_cogs'])}")
print(f"ВАЛОВАЯ ПРИБЫЛЬ:                {fmt(base['gross_profit'])}")
print(f"Аренда:                         {fmt(RENT_PER_MONTH)}")
print(f"ФОТ (с налогами):               {fmt(PAYROLL_TOTAL)}")
print(f"Маркетинг:                      {fmt(MARKETING)}")
print(f"Коммуналка/прочее:              {fmt(UTILITIES + LICENSES_OTHER)}")
print(f"Роялти Surf Coffee (5%):        {fmt(base['franchise_fee'])}")
print("─" * 60)
print(f"ОПЕРАЦИОННАЯ ПРИБЫЛЬ:           {fmt(base['operating_profit'])}")
print(f"Операционная маржа:             {base['margin_pct']:.1f}%")

In [ ]:
# Используем формулу:
# BEP (в чеках в день) = постоянные_расходы / (средний_чек * маржинальность_по_чеку * дней_в_месяце)

# Маржинальность одного чека (после COGS и роялти, до постоянных)
margin_per_check = AVG_CHECK * (
    1 - (COFFEE_REVENUE_SHARE * COFFEE_COGS_PCT + (1 - COFFEE_REVENUE_SHARE) * FOOD_COGS_PCT)
    - FRANCHISE_FEE_PCT
)

fixed_total = RENT_PER_MONTH + PAYROLL_TOTAL + MARKETING + UTILITIES + LICENSES_OTHER

bep_checks_month = fixed_total / margin_per_check
bep_checks_day = bep_checks_month / DAYS_PER_MONTH
bep_revenue_month = bep_checks_month * AVG_CHECK

print("=" * 60)
print("ТОЧКА БЕЗУБЫТОЧНОСТИ (BEP)")
print("=" * 60)
print(f"Постоянные расходы в месяц:     {fmt(fixed_total)}")
print(f"Маржа с одного чека:            {fmt(margin_per_check)}")
print(f"BEP (чеков в месяц):            {bep_checks_month:.0f}")
print(f"BEP (чеков в день):             {bep_checks_day:.0f}")
print(f"BEP (выручка в месяц):          {fmt(bep_revenue_month)}")
print(f"\nДля выхода в ноль нужно ~{bep_checks_day:.0f} чеков в день.")
print(f"Базовый прогноз: {DRINKS_PER_DAY_BASE} чеков → запас {(DRINKS_PER_DAY_BASE - bep_checks_day)/DRINKS_PER_DAY_BASE*100:.0f}% над BEP.")

In [ ]:
scenarios = {
    "Пессимистичный": calculate_monthly_pnl(160),  # медленный старт
    "Базовый":        calculate_monthly_pnl(240),  # норма для локации
    "Оптимистичный":  calculate_monthly_pnl(320),  # пик потока студентов
}

# Собираем сравнительную таблицу
df_scenarios = pd.DataFrame({
    name: {
        "Чеков в день":             s["checks_per_day"],
        "Выручка/мес, ₽":           round(s["revenue"]),
        "COGS, ₽":                  round(s["total_cogs"]),
        "Валовая прибыль, ₽":       round(s["gross_profit"]),
        "Постоянные расходы, ₽":    round(s["fixed_costs"]),
        "Роялти, ₽":                round(s["franchise_fee"]),
        "Операц. прибыль/мес, ₽":   round(s["operating_profit"]),
        "Маржа, %":                 round(s["margin_pct"], 1),
        "Операц. прибыль/год, ₽":   round(s["operating_profit"] * 12),
        "Окупаемость, мес.":        round(STARTUP_INVESTMENT / s["operating_profit"], 1) if s["operating_profit"] > 0 else None,
    }
    for name, s in scenarios.items()
}).T

print("СРАВНЕНИЕ СЦЕНАРИЕВ:")
print(df_scenarios.to_string())

# Сохраняем модель в CSV
df_scenarios.to_csv("data/processed/financial_model.csv", encoding="utf-8-sig")
print("\nСохранено: data/processed/financial_model.csv")

In [ ]:
base = scenarios["Базовый"]
labels = ["Аренда", "ФОТ (с налогами)", "COGS", "Роялти", "Маркетинг", "Коммуналка/прочее"]
values = [
    RENT_PER_MONTH,
    PAYROLL_TOTAL,
    base["total_cogs"],
    base["franchise_fee"],
    MARKETING,
    UTILITIES + LICENSES_OTHER,
]
colors = ["#4A90D9", "#E8A838", "#D95B5B", "#9B59B6", "#5BA85A", "#7F8C8D"]

fig, ax = plt.subplots(figsize=(8, 5))
wedges, texts, autotexts = ax.pie(
    values, labels=labels, autopct="%1.1f%%", colors=colors,
    startangle=90, pctdistance=0.78,
    textprops={"fontsize": 10},
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight("bold")
    at.set_color("white")

ax.set_title(f"Структура ежемесячных расходов\n(базовый сценарий, {fmt(sum(values))})",
             fontsize=12, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig("visuals/13_cost_structure.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: visuals/13_cost_structure.png")

In [ ]:
names = list(scenarios.keys())
profits = [s["operating_profit"] for s in scenarios.values()]
revenues = [s["revenue"] for s in scenarios.values()]

fig, ax = plt.subplots(figsize=(9, 5))
x = np.arange(len(names))
width = 0.35

b1 = ax.bar(x - width/2, revenues, width, label="Выручка", color="#4A90D9")
b2 = ax.bar(x + width/2, profits, width, label="Операц. прибыль", color="#E8A838")

ax.set_title("Выручка и операционная прибыль по сценариям (в месяц)",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xticks(x)
ax.set_xticklabels([f"{n}\n({s['checks_per_day']} чеков/день)"
                    for n, s in scenarios.items()], fontsize=10)
ax.set_ylabel("Сумма, ₽", fontsize=11)
ax.legend(fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)

# Подписи на барах
for bar in b1:
    h = bar.get_height()
    ax.annotate(fmt(h), (bar.get_x() + bar.get_width()/2, h),
                ha="center", va="bottom", fontsize=9, fontweight="bold")
for bar in b2:
    h = bar.get_height()
    label = fmt(h) if h > 0 else f"-{fmt(abs(h))}"
    ax.annotate(label, (bar.get_x() + bar.get_width()/2, h),
                ha="center", va="bottom" if h > 0 else "top",
                fontsize=9, fontweight="bold",
                color="#E8A838" if h > 0 else "#D95B5B")

# Линия нуля
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.savefig("visuals/14_scenarios_pnl.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: visuals/14_scenarios_pnl.png")

In [ ]:
checks_range = np.arange(50, 351, 10)
revenues_curve = [calculate_monthly_pnl(c)["revenue"] for c in checks_range]
costs_curve = [calculate_monthly_pnl(c)["total_cogs"] + calculate_monthly_pnl(c)["franchise_fee"] + fixed_total
               for c in checks_range]
profit_curve = [calculate_monthly_pnl(c)["operating_profit"] for c in checks_range]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(checks_range, revenues_curve, "-", color="#4A90D9", linewidth=2.5, label="Выручка")
ax.plot(checks_range, costs_curve, "-", color="#D95B5B", linewidth=2.5, label="Общие расходы")
ax.fill_between(checks_range, revenues_curve, costs_curve,
                where=np.array(revenues_curve) >= np.array(costs_curve),
                color="#5BA85A", alpha=0.2, label="Прибыль")
ax.fill_between(checks_range, revenues_curve, costs_curve,
                where=np.array(revenues_curve) < np.array(costs_curve),
                color="#D95B5B", alpha=0.2, label="Убыток")

# BEP
ax.axvline(bep_checks_day, color="black", linestyle="--", linewidth=1.5, alpha=0.7)
ax.annotate(f"BEP: {bep_checks_day:.0f} чеков/день\n({fmt(bep_revenue_month)}/мес)",
            xy=(bep_checks_day, bep_revenue_month),
            xytext=(bep_checks_day + 15, bep_revenue_month - 500_000),
            fontsize=10, fontweight="bold",
            arrowprops=dict(arrowstyle="->", color="black"))

# Маркируем сценарии
for name, sc in scenarios.items():
    ax.scatter([sc["checks_per_day"]], [sc["revenue"]], s=120, zorder=5,
               edgecolor="black", linewidth=1)
    ax.annotate(name, (sc["checks_per_day"], sc["revenue"]),
                textcoords="offset points", xytext=(5, 12), fontsize=10, fontweight="bold")

ax.set_title("Точка безубыточности и сценарии Surf Coffee",
             fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Чеков в день", fontsize=11)
ax.set_ylabel("Сумма в месяц, ₽", fontsize=11)
ax.legend(loc="upper left", fontsize=10)
ax.grid(linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/15_breakeven.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: visuals/15_breakeven.png")

In [ ]:
# Реалистичная модель: трафик нарастает в первые месяцы

# Доля от целевого трафика по месяцам (ramp-up curve)
ramp_up = [0.40, 0.55, 0.70, 0.80, 0.88, 0.95, 1.00, 1.05, 1.10, 1.15, 1.20, 1.20]
months = [f"М{i+1}" for i in range(12)]

monthly_data = []
cum_profit = -STARTUP_INVESTMENT  # стартовые инвестиции — минусом

for i, ratio in enumerate(ramp_up):
    checks = int(DRINKS_PER_DAY_BASE * ratio)
    pnl = calculate_monthly_pnl(checks)
    cum_profit += pnl["operating_profit"]
    monthly_data.append({
        "Месяц": months[i],
        "Чеков/день": checks,
        "Выручка": pnl["revenue"],
        "Операц. прибыль": pnl["operating_profit"],
        "Накопленный денежный поток": cum_profit,
    })

df_year = pd.DataFrame(monthly_data)
print("ПРОГНОЗ НА 12 МЕСЯЦЕВ (с учётом ramp-up):")
print(df_year.to_string(index=False))

# Месяц выхода в плюс по cash flow
profitable_month = df_year[df_year["Накопленный денежный поток"] > 0].index.min()
if pd.notna(profitable_month):
    print(f"\nВыход в плюс по накопленному кэшу: М{profitable_month + 1}")
else:
    print(f"\nЗа первый год кэш не выходит в плюс. Конец года: {fmt(df_year['Накопленный денежный поток'].iloc[-1])}")

# Сохраняем прогноз
df_year.to_csv("data/processed/financial_forecast_year1.csv",
               index=False, encoding="utf-8-sig")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

cumcash = df_year["Накопленный денежный поток"].values
months_x = np.arange(12)

# Цвета: красный при минусе, зелёный при плюсе
colors_cf = ["#D95B5B" if v < 0 else "#5BA85A" for v in cumcash]
bars = ax.bar(months_x, cumcash, color=colors_cf, edgecolor="white", linewidth=1.5)

ax.set_xticks(months_x)
ax.set_xticklabels(df_year["Месяц"])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Накопленный денежный поток по месяцам (год 1)\n(старт минусом по инвестициям)",
             fontsize=12, fontweight="bold", pad=12)
ax.set_ylabel("Накопленный кэш, ₽", fontsize=11)
ax.set_xlabel("Месяц", fontsize=11)

for bar, val in zip(bars, cumcash):
    y = bar.get_height()
    va = "bottom" if y >= 0 else "top"
    offset = 50_000 if y >= 0 else -50_000
    ax.text(bar.get_x() + bar.get_width()/2, y + offset,
            fmt(val), ha="center", va=va, fontsize=8, fontweight="bold")

ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig("visuals/16_cash_flow_year1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Сохранено: visuals/16_cash_flow_year1.png")

In [ ]:
print(f"""
╔══════════════════════════════════════════════════════════════════╗
║              ВЫВОДЫ ПО ФИНАНСОВОЙ МОДЕЛИ                       ║
╚══════════════════════════════════════════════════════════════════╝

ВХОДНЫЕ ДАННЫЕ (из опроса и рынка):
• Средний чек:                    {fmt(AVG_CHECK)}
• Базовый трафик:                 {DRINKS_PER_DAY_BASE} чеков/день
• Стартовые инвестиции:           {fmt(STARTUP_INVESTMENT)}
• Постоянные расходы в месяц:     {fmt(fixed_total)}

КЛЮЧЕВЫЕ ПОКАЗАТЕЛИ (базовый сценарий):
• Выручка в месяц:                {fmt(scenarios["Базовый"]["revenue"])}
• Операционная прибыль в месяц:   {fmt(scenarios["Базовый"]["operating_profit"])}
• Операционная маржа:             {scenarios["Базовый"]["margin_pct"]:.1f}%
• Операционная прибыль за год:    {fmt(scenarios["Базовый"]["operating_profit"] * 12)}

ТОЧКА БЕЗУБЫТОЧНОСТИ:
• {bep_checks_day:.0f} чеков в день
• {fmt(bep_revenue_month)} выручки в месяц
• Запас прочности в базовом сценарии: {(DRINKS_PER_DAY_BASE - bep_checks_day)/DRINKS_PER_DAY_BASE*100:.0f}%

ОКУПАЕМОСТЬ:
• Базовый сценарий:        ~{STARTUP_INVESTMENT / scenarios["Базовый"]["operating_profit"]:.1f} месяцев
• Оптимистичный:           ~{STARTUP_INVESTMENT / scenarios["Оптимистичный"]["operating_profit"]:.1f} месяцев
• Пессимистичный:          {"убыток — не окупается" if scenarios["Пессимистичный"]["operating_profit"] < 0 else f"~{STARTUP_INVESTMENT / scenarios['Пессимистичный']['operating_profit']:.1f} мес."}

РИСКИ:
• Главный риск — низкий стартовый трафик (BEP = {bep_checks_day:.0f} чеков/день).
  При 150 чеков/день модель убыточна, нужен план Б по маркетингу.
• Высокая доля ФОТ ({PAYROLL_TOTAL/fixed_total*100:.0f}% постоянных расходов) —
  при просадке трафика её сложно быстро сократить.
• Зависимость от учебного календаря: каникулы, летние месяцы могут
  снизить трафик на 30-40%.

РЕКОМЕНДАЦИИ:
1. Заложить маркетинг-старт (флаеры/SMM в студенческих чатах ВШЭ).
2. Сформировать программу лояльности для студентов (5-10 кофе и
   1 бесплатный) — повышает retention и средний чек.
3. На каникулах сокращать смены и фокусироваться на сотрудниках ВШЭ
   и людях, которые работают из кофейни.
4. Заложить буфер ~1.5 млн ₽ оборотных средств на первые 4 месяца.
""")